In [ ]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path.cwd().parent if (Path.cwd().parent / "sample_data").is_dir() else Path.cwd()
DATA_DIR = REPO_ROOT / "sample_data"
df = pd.read_excel(DATA_DIR / "art_data.xlsx")

# 保存成 csv
df.to_csv(DATA_DIR / "art_data.csv", index=False, encoding="utf-8-sig")

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import mean_squared_error, make_scorer


df.columns = [c.strip() for c in df.columns]
df = df.replace({"na": np.nan, "NA": np.nan, "N/A": np.nan, "": np.nan})

#set up x and y
y_col = "positive_emotion"
drop_cols = ["student_id", "text", "continue_participation",'painting_path',
             'processed_painting_path','negative_release',
             'others']  # delete the unwanted columns.

# 把除 words 外的列尽量转成数值（非数值会变 NaN）
for c in df.columns:
    if c != "words":
        df[c] = pd.to_numeric(df[c], errors="coerce")

# 丢掉 y 缺失行
df = df.dropna(subset=[y_col])

y = df[y_col].astype(float)
X = df.drop(columns=[y_col] + drop_cols, errors="ignore")

# 4) 回归管线：缺失值填补 + 缺失指示器 + 标准化 + ElasticNet/Lasso
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

rmse_scorer = make_scorer(rmse, greater_is_better=False)

model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),  # add_indicator: 把“是否缺失”也作为特征
    ("scaler", StandardScaler()),
    ("reg", ElasticNetCV(
        l1_ratio=[.1, .3, .5, .7, .9, 1.0],  # 1.0 就是 Lasso，更强的变量筛选
        cv=5,
        random_state=42,
        max_iter=20000
    ))
])

# 5) 交叉验证评估
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(
    model, X, y, cv=cv,
    scoring={
        "r2": "r2",
        "neg_mae": "neg_mean_absolute_error",
        "neg_rmse": rmse_scorer
    }
)

print("CV R2    :", scores["test_r2"].mean(), "+/-", scores["test_r2"].std())
print("CV MAE   :", -scores["test_neg_mae"].mean(), "+/-", scores["test_neg_mae"].std())
print("CV RMSE  :", -scores["test_neg_rmse"].mean(), "+/-", scores["test_neg_rmse"].std())

model.fit(X, y)
reg = model.named_steps["reg"]


In [ ]:
imputer = model.named_steps["imputer"]
reg = model.named_steps["reg"]

# 关键：直接取 imputer 变换后的真实特征名
feat_names = imputer.get_feature_names_out(X.columns)
coef = reg.coef_

print("len(feat_names) =", len(feat_names))
print("len(coef)       =", len(coef))

coef_df = pd.DataFrame({
    "feature": feat_names,
    "coef": coef
}).sort_values("coef", ascending=False)

print("\nTop positive predictors:")
print(coef_df.head(10).to_string(index=False))

print("\nTop negative predictors:")
print(coef_df.tail(10).to_string(index=False))

In [ ]:
use = df.iloc[:, [1, 5, 6, 7, 8 ,4, 10]].copy()
use.columns = ["level", "write", "imagine", "ref", "color",'neg','pos']
use.head()


In [ ]:
use = use.dropna(subset=[use.columns[0]])
use = use[use.iloc[:,0].isin([1,2,3])]

for c in use.columns[1:]:
    use[c] = pd.to_numeric(use[c], errors="coerce")

use.shape


In [ ]:
use.groupby(use.columns[0])[use.columns[1:]].mean().round(2)


In [ ]:
use[use.columns[0]].value_counts().sort_index()


In [ ]:
from scipy.stats import kruskal

level = use.columns[0]
write, imagine, ref, color = use.columns[1], use.columns[2], use.columns[3], use.columns[4]

# ① 写下情绪
p_write = kruskal(
    use.loc[use[level]==1, write].dropna(),
    use.loc[use[level]==2, write].dropna(),
    use.loc[use[level]==3, write].dropna()
).pvalue

# ② 想象构图
p_imagine = kruskal(
    use.loc[use[level]==1, imagine].dropna(),
    use.loc[use[level]==2, imagine].dropna(),
    use.loc[use[level]==3, imagine].dropna()
).pvalue

# ③ 参考图片
p_ref = kruskal(
    use.loc[use[level]==1, ref].dropna(),
    use.loc[use[level]==2, ref].dropna(),
    use.loc[use[level]==3, ref].dropna()
).pvalue

# ④ 颜色表达
p_color = kruskal(
    use.loc[use[level]==1, color].dropna(),
    use.loc[use[level]==2, color].dropna(),
    use.loc[use[level]==3, color].dropna()
).pvalue

p_write, p_imagine, p_ref, p_color


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kruskal

# 假设 use 已存在
level = use.columns[0]
write, imagine, ref, color = use.columns[1], use.columns[2], use.columns[3], use.columns[4]

steps = [
    ("wring_down_emotions", write),
    ("imagine_composition", imagine),
    ("reference_to_the_picture", ref),
    ("color_reference", color),
]

def kruskal_p_and_eps2(df, level_col, value_col, groups=(1,2,3)):
    # 取出各组数据
    arrays = []
    ns = []
    for g in groups:
        a = df.loc[df[level_col] == g, value_col].dropna().values
        arrays.append(a)
        ns.append(len(a))

    # Kruskal-Wallis
    res = kruskal(*arrays, nan_policy='omit')
    H = res.statistic
    p = res.pvalue

    # 效应量 epsilon-squared（Kruskal 常用近似）
    # eps2 = (H - k + 1) / (n - k)
    k = len(groups)
    n = int(np.sum(ns))
    eps2 = (H - k + 1) / (n - k) if (n - k) > 0 else np.nan
    eps2 = max(0.0, float(eps2)) if np.isfinite(eps2) else np.nan  # 防止出现负数

    return p, H, eps2, ns

rows = []
for name, col in steps:
    p, H, eps2, ns = kruskal_p_and_eps2(use, level, col, groups=(1,2,3))
    rows.append({"step": name, "p": p, "H": H, "eps2": eps2, "n_by_group": ns})

res_df = pd.DataFrame(rows)

# --- 图1：p-value 强度对比（推荐）---
# 用 -log10(p) 更直观；p=0 时做个下限保护
p_safe = res_df["p"].clip(lower=1e-300)
res_df["neglog10p"] = -np.log10(p_safe)

plt.figure(figsize=(8, 4.5))
plt.bar(res_df["step"], res_df["neglog10p"])
plt.ylabel(r"$-\log_{10}(p)$  (larger = stronger difference evidence)")
plt.title("Comparison of significance strength across four steps by level (Kruskal–Wallis)")
plt.xticks(rotation=0)

# 显著性阈值线：p=0.05/0.01/0.001
for p_thr in [0.05, 0.01, 0.001]:
    y = -np.log10(p_thr)
    plt.axhline(y, linestyle="--", linewidth=1)
    plt.text(-0.4, y, f" p={p_thr}", va="bottom")

# 在柱子上标注原始 p 值
for i, (val, pval) in enumerate(zip(res_df["neglog10p"], res_df["p"])):
    plt.text(i, val, f"{pval:.2e}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

# --- 图2：可选：效应量（更接近“差异有多大”）---
plt.figure(figsize=(8, 4.5))
plt.bar(res_df["step"], res_df["eps2"])
plt.ylabel(r"Effect size $\varepsilon^2$ (approx.)")
plt.title("Effect size across four steps by level (ε², Kruskal–Wallis approximation)")
plt.xticks(rotation=0)

for i, v in enumerate(res_df["eps2"]):
    if np.isfinite(v):
        plt.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

res_df


In [ ]:
print("writing down emotionsp =", p_write)
print("imagine composition p =", p_imagine)
print("reference to picture p =", p_ref)
print("color expression p =", p_color)


In [ ]:
!pip -q install scikit-posthocs
import scikit_posthocs as sp

# 以“想象构图”为例（你也可以换成 write/ref/color）
tmp = use[[level, imagine]].dropna()
sp.posthoc_dunn(tmp, val_col=imagine, group_col=level, p_adjust="holm")


In [ ]:
import matplotlib.pyplot as plt

# 准备三组数据
g1 = use.loc[use[level]==1, imagine].dropna()
g2 = use.loc[use[level]==2, imagine].dropna()
g3 = use.loc[use[level]==3, imagine].dropna()

plt.figure(figsize=(6,4))

# 箱线图
plt.boxplot([g1, g2, g3], tick_labels=["Level 1","Level 2","Level 3"], showfliers=False,
            medianprops=dict(color="purple", linewidth=3))

# 抖动散点（让每个点不重叠）
import numpy as np
for i, g in enumerate([g1, g2, g3], start=1):
    x = np.random.normal(i, 0.04, size=len(g))
    plt.scatter(x, g, alpha=0.25)

plt.ylabel("Importance rating: Imagine composition (1–5)")
plt.title("Imagine composition importance by art level")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 你要画的三个步骤（想加 imagine 也行）
plots = [
    ("Writing down emotions", write),
    ("Reference to picture", ref),
    ("Color expression", color),
]

for title, col in plots:
    g1 = use.loc[use[level]==1, col].dropna()
    g2 = use.loc[use[level]==2, col].dropna()
    g3 = use.loc[use[level]==3, col].dropna()

    plt.figure(figsize=(6,4))

    # boxplot（修正 warning：tick_labels）
    plt.boxplot([g1, g2, g3],
                tick_labels=["Level 1","Level 2","Level 3"],
                showfliers=False,
               medianprops=dict(color="red", linewidth=3))

    # jitter points
    for i, g in enumerate([g1, g2, g3], start=1):
        x = np.random.normal(i, 0.04, size=len(g))
        plt.scatter(x, g, alpha=0.25)

    plt.ylabel("Importance rating (1–5)")
    plt.title(f"{title} importance by art level")
    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 准备三组数据
g1 = use.loc[use[level]==1, imagine].dropna()
g2 = use.loc[use[level]==2, imagine].dropna()
g3 = use.loc[use[level]==3, imagine].dropna()

plt.figure(figsize=(6,4))

# 箱线图
plt.boxplot([g1, g2, g3], tick_labels=["Level 1","Level 2","Level 3"], showfliers=False,
            medianprops=dict(color="red", linewidth=3))

# 抖动散点（让每个点不重叠）
import numpy as np
for i, g in enumerate([g1, g2, g3], start=1):
    x = np.random.normal(i, 0.04, size=len(g))
    plt.scatter(x, g, alpha=0.25)

plt.ylabel("Importance rating: Imagine composition (1–5)")
plt.title("Imagine composition importance by art level")
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plots = [
    ("Writing down emotions", write),
    ("Imagine composition", imagine),
    ("Reference to picture", ref),
    ("Color expression", color),
]

for title, col in plots:
    g1 = use.loc[use[level]==1, col].dropna()
    g2 = use.loc[use[level]==2, col].dropna()
    g3 = use.loc[use[level]==3, col].dropna()

    data = [g1, g2, g3]

    plt.figure(figsize=(6,4))

    # 1) violin：看分布形状（哪里更“厚”代表更多人打这个分）
    plt.violinplot(data, showmeans=False, showmedians=False, showextrema=False)

    # 2) box：看中位数/IQR（更直观）
    plt.boxplot(data, tick_labels=["Level 1","Level 2","Level 3"],
                showfliers=False, widths=0.25,
                medianprops=dict(color="red", linewidth=3)
                )

    # 3) jitter：看每个样本点（让分布更真实）
    for i, g in enumerate(data, start=1):
        x = np.random.normal(i, 0.04, size=len(g))
        plt.scatter(x, g, alpha=0.20, s=10)

    plt.title(f"{title} importance by art level")
    plt.ylabel("Importance rating (1–5)")
    plt.tight_layout()
    plt.show()


In [ ]:
use["neg"] = pd.to_numeric(df.iloc[:, 4], errors="coerce")
use["pos"] = pd.to_numeric(df.iloc[:,10], errors="coerce")
neg, pos = "neg", "pos"

use

In [ ]:
from scipy.stats import spearmanr
u1 = use[use[level]==1][[write, imagine, ref, color, neg, pos]].dropna()
print("Level 1 n =", len(u1))
print("write vs neg:", spearmanr(u1[write], u1[neg]))
print("write vs pos:", spearmanr(u1[write], u1[pos]))
print("imagine vs neg:", spearmanr(u1[imagine], u1[neg]))
print("imagine vs pos:", spearmanr(u1[imagine], u1[pos]))
print("ref vs neg:", spearmanr(u1[ref], u1[neg]))
print("ref vs pos:", spearmanr(u1[ref], u1[pos]))
print("color vs neg:", spearmanr(u1[color], u1[neg]))
print("color vs pos:", spearmanr(u1[color], u1[pos]))


In [ ]:
u2 = use[use[level]==2][[write, imagine, ref, color, neg, pos]].dropna()

print("Level 2 n =", len(u2))
print("write vs neg:", spearmanr(u2[write], u2[neg]))
print("write vs pos:", spearmanr(u2[write], u2[pos]))
print("imagine vs neg:", spearmanr(u2[imagine], u2[neg]))
print("imagine vs pos:", spearmanr(u2[imagine], u2[pos]))
print("ref vs neg:", spearmanr(u2[ref], u2[neg]))
print("ref vs pos:", spearmanr(u2[ref], u2[pos]))
print("color vs neg:", spearmanr(u2[color], u2[neg]))
print("color vs pos:", spearmanr(u2[color], u2[pos]))


In [ ]:
u3 = use[use[level]==3][[write, imagine, ref, color, neg, pos]].dropna()

print("Level 3 n =", len(u3))
print("write vs neg:", spearmanr(u3[write], u3[neg]))
print("write vs pos:", spearmanr(u3[write], u3[pos]))
print("imagine vs neg:", spearmanr(u3[imagine], u3[neg]))
print("imagine vs pos:", spearmanr(u3[imagine], u3[pos]))
print("ref vs neg:", spearmanr(u3[ref], u3[neg]))
print("ref vs pos:", spearmanr(u3[ref], u3[pos]))
print("color vs neg:", spearmanr(u3[color], u3[neg]))
print("color vs pos:", spearmanr(u3[color], u3[pos]))


In [ ]:
def one_row(u, step, outcome):
    r, p = spearmanr(u[step], u[outcome])
    return round(r, 3), p

rows = []

for lvl, u in [(1,u1),(2,u2),(3,u3)]:
    r1,p1 = one_row(u, write, neg)
    r2,p2 = one_row(u, write, pos)
    r3,p3 = one_row(u, ref, neg)
    r4,p4 = one_row(u, ref, pos)

    rows.append([lvl, len(u), r1, p1, r2, p2, r3, p3, r4, p4])

table = pd.DataFrame(rows, columns=[
    "level","n",
    "write~neg_r","write~neg_p",
    "write~pos_r","write~pos_p",
    "ref~neg_r","ref~neg_p",
    "ref~pos_r","ref~pos_p"
])
table


In [ ]:
from scipy.stats import spearmanr
import pandas as pd

# 你这几个变量应该在前面已经定义好了（如果没定义，按你的列名/列号定义一下）
# level = use.columns[0]
# write, imagine, ref, color = use.columns[1], use.columns[2], use.columns[3], use.columns[4]
# neg = "坏情绪释放程度5"
# pos = "积极情绪7"

steps = {
    "write": write,
    "imagine": imagine,
    "ref": ref,
    "color": color
}

outcomes = {
    "neg": neg,
    "pos": pos
}

rows = []

for lvl in [1, 2, 3]:
    # 为了让同一组里所有相关性用同一批样本：把四步骤+两效果都不缺失的行保留下来
    cols_need = list(steps.values()) + list(outcomes.values())
    u = use[use[level] == lvl][cols_need].dropna()

    row = {"level": lvl, "n": len(u)}

    for s_key, s_col in steps.items():
        for o_key, o_col in outcomes.items():
            r, p = spearmanr(u[s_col], u[o_col])
            row[f"{s_key}~{o_key}_r"] = round(r, 3)
            row[f"{s_key}~{o_key}_p"] = p

    rows.append(row)

table = pd.DataFrame(rows)
table


In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

# ===== 数据准备（沿用你之前逻辑）=====

df.columns = [c.strip() for c in df.columns]
df = df.replace({"na": np.nan, "NA": np.nan, "N/A": np.nan, "": np.nan, " ": np.nan})

for c in df.columns:
    if c != "words":
        df[c] = pd.to_numeric(df[c], errors="coerce")

y_col = "positive mood"
drop_cols = ["student_number", "words", "future", "release_scale", "other_factor"]

df = df.dropna(subset=[y_col])
y = df[y_col].astype(float)
X = df.drop(columns=[y_col] + drop_cols, errors="ignore")
X = X.select_dtypes(include=[np.number])
X = X.dropna(axis=1, how="all")
X = X.loc[:, X.nunique(dropna=True) > 1]

# ===== 评估指标 =====
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))
rmse_scorer = make_scorer(rmse, greater_is_better=False)

# ===== 模型：HistGradientBoostingRegressor（能处理 NaN）=====
hgb = HistGradientBoostingRegressor(
    learning_rate=0.05,
    max_depth=3,
    max_iter=400,
    random_state=42
)

# ===== 5折CV评估（预测可靠性）=====
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(
    hgb, X, y, cv=cv,
    scoring={"r2":"r2","neg_mae":"neg_mean_absolute_error","neg_rmse":rmse_scorer}
)

print("HGB CV R2   :", scores["test_r2"].mean(), "+/-", scores["test_r2"].std())
print("HGB CV MAE  :", -scores["test_neg_mae"].mean(), "+/-", scores["test_neg_mae"].std())
print("HGB CV RMSE :", -scores["test_neg_rmse"].mean(), "+/-", scores["test_neg_rmse"].std())

# ===== 变量重要性：Permutation Importance（解释可靠性）=====
# 这里用一次 train/test split 做示例；更严谨可在每个fold上算再平均
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
hgb.fit(Xtr, ytr)

perm = permutation_importance(
    hgb, Xte, yte,
    n_repeats=30,
    random_state=42,
    scoring="r2"
)

pi = pd.DataFrame({
    "feature": X.columns,
    "perm_r2_drop_mean": perm.importances_mean,
    "perm_r2_drop_std": perm.importances_std
}).sort_values("perm_r2_drop_mean", ascending=False)

print("\nPermutation importance (bigger = more important):")
print(pi.to_string(index=False))


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# 1) 读数据
df=pd.read_csv(DATA_DIR / 'art_data.csv')
df.columns = [c.strip() for c in df.columns]
df = df.replace({"na": np.nan, "NA": np.nan, "N/A": np.nan, "": np.nan, " ": np.nan})
df = df.dropna(how="all")

# 2) 除 words 外尽量转成数值
for c in df.columns:
    if c != "text":
        df[c] = pd.to_numeric(df[c], errors="coerce")

# 3) 设置 y 与 blocks（你要排除的列不进入 X）
y_col = "positive_emotion"
drop_cols = ["student_id", "text", "continue_participation", "negative_release", "others"]

block1 = ["drawing_skills"]
block3 = ["write_emotions", "imagine_composition", "examples", "color_expression", "level_difficulty"]
block2 = ["drawing_participation", "enjoy_drawing"]

# 4) 为了公平比较ΔR²：用“全模型所需列”的 complete cases
all_predictors = block1 + block2 + block3
data = df[[y_col] + all_predictors].dropna().copy()

# statsmodels 公式不支持列名有空格：重命名 y
data = data.rename(columns={"positive mood": "positive_mood"})
y = "positive_emotion"

# 5) 建立分层模型
f1 = f"{y} ~ " + " + ".join(block1)
f2 = f"{y} ~ " + " + ".join(block1 + block2)
f3 = f"{y} ~ " + " + ".join(block1 + block2 + block3)

m1 = smf.ols(f1, data=data).fit()
m2 = smf.ols(f2, data=data).fit()
m3 = smf.ols(f3, data=data).fit()

# 6) R² 与 ΔR²
comp = pd.DataFrame({
    "Model": ["M1: art level", "M2: + engagement + interest ", "M3: + step factors "],
    "N": [int(m1.nobs), int(m2.nobs), int(m3.nobs)],
    "R2": [m1.rsquared, m2.rsquared, m3.rsquared],
    "Adj_R2": [m1.rsquared_adj, m2.rsquared_adj, m3.rsquared_adj],
})
comp["Delta_R2"] = comp["R2"].diff()
print(comp)

# 7) 增量显著性（F-change）
print("\nF-change tests (nested model comparison):")
print(anova_lm(m1, m2, m3))

# 8)（推荐）输出全模型的“标准化β”，方便比较谁更重要
cols_to_z = [y] + all_predictors
z = data[cols_to_z].apply(lambda s: (s - s.mean())/s.std(ddof=0))
m3_std = smf.ols(f3, data=z).fit()

std_table = pd.DataFrame({
    "term": m3_std.params.index,
    "beta_std": m3_std.params.values,
    "p_value": m3_std.pvalues.values,
    "ci_low": m3_std.conf_int()[0].values,
    "ci_high": m3_std.conf_int()[1].values
}).query("term != 'Intercept'").sort_values("beta_std", ascending=False)

print("\nStandardized betas (Model 3):")
print(std_table.to_string(index=False))


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

df = df.rename(columns={"positive_emotion": "positive_emotion"})

# 四步变量（你前面用的列名）
steps = ["write_emotions", "imagine_composition", "examples", "color_expression"]

# 可选控制变量（与你之前分层回归一致）
controls = ["drawing_skills", "level_difficulty", "drawing_participation", "enjoy_drawing"]

# 只保留必要列，并去缺失（避免不同模型用不同样本）
need_cols = ["positive_emotion"] + steps + controls
data = df[need_cols].dropna().copy()

print("N used for RQ4:", len(data))

# =========================
# 1) 计算“有效性权重” E
#    方法：用标准化回归系数 beta_std 当作每步的有效性权重
#    你可以选：A) 只用 steps 做模型；B) steps + controls 做模型（更像“净效应”）
# =========================

def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

# 选择 A 或 B（推荐 B：更接近“控制其他因素后，这一步仍然有效”）
USE_CONTROLS_FOR_WEIGHTS = True

if USE_CONTROLS_FOR_WEIGHTS:
    cols_for_weights = ["positive_emotion"] + steps + controls
    formula = "positive_emotion ~ " + " + ".join(steps + controls)
else:
    cols_for_weights = ["positive_emotion"] + steps
    formula = "positive_emotion ~ " + " + ".join(steps)

z = data[cols_for_weights].apply(zscore)

m = smf.ols(formula, data=z).fit()
beta = m.params[steps]  # 标准化 beta（只取四步）
print("\nStandardized betas (effectiveness weights):")
print(beta)

# 把 beta 转成“非负权重向量”（更直观：只强调正向有效步骤）
# 如果你想允许负向（惩罚），可以把下面 clamp 去掉
beta_pos = beta.clip(lower=0)

# 如果全是0（极端情况），避免除0
if beta_pos.sum() == 0:
    raise ValueError("All step betas are <= 0; cannot form positive effectiveness weights.")
E = (beta_pos / beta_pos.sum()).values  # 归一化成和为1的权重
E_names = steps

print("\nNormalized effectiveness weights E (sum=1):")
for name, w in zip(E_names, E):
    print(f"{name}: {w:.4f}")

# =========================
# 2) 计算每个参与者的“偏好权重” w_i（重要性分配）
#    w_i = step_score / sum(step_scores)
# =========================
S = data[steps].copy()

row_sum = S.sum(axis=1)
# 防止有人四步全0（理论上不会，但保险）
row_sum = row_sum.replace(0, np.nan)

W = S.div(row_sum, axis=0)  # 每行归一化
W = W.fillna(0)

# =========================
# 3) 计算 alignment（对齐度）
#    给你两种方式（选一个就行）
#    A) Weighted effectiveness index: alignment = Σ w_i * E
#    B) Cosine similarity: cos(w_i, E)
# =========================

# A) 加权有效偏好指数（最直观）
data["alignment_index"] = W.values @ E

# B) Cosine similarity（范围 0~1，越大越对齐）
E_vec = E / np.linalg.norm(E)
W_norm = np.linalg.norm(W.values, axis=1)
# 防止除0
W_norm = np.where(W_norm == 0, 1e-12, W_norm)
data["alignment_cosine"] = (W.values @ E_vec) / W_norm

print("\nAlignment summary:")
print(data[["alignment_index", "alignment_cosine"]].describe())

# =========================
# 4) 用回归检验：对齐度是否与 positive_mood 更高相关？
#    你可以：
#    - 不控制：positive_mood ~ alignment
#    - 控制：positive_mood ~ alignment + ability + interest + ...
# =========================

# 选择用哪个 alignment
ALIGN_COL = "alignment_index"   # 或 "alignment_cosine"

# (1) 简单回归
r1 = smf.ols(f"positive_emotion ~ {ALIGN_COL}", data=data).fit()
print("\n[Model R1] positive_emotion~ alignment")
print(r1.summary().tables[1])

# (2) 控制变量回归（推荐用于论文表述）
r2 = smf.ols(f"positive_emotion ~ {ALIGN_COL} + " + " + ".join(controls), data=data).fit()
print("\n[Model R2] positive_emotion ~ alignment + controls")
print(r2.summary().tables[1])

# 你最关心的：ALIGN_COL 这一行的 coef 与 p-value
# coef > 0 且显著：对齐越高，mood 越高（支持“偏好-有效性一致性”）
# 不显著：偏好是否对齐不影响 mood（或效果很弱）

# =========================
# 5) 组别层面的“错配”（可选：按 ability_of_art 分组，看偏好排名 vs 有效性排名）
# =========================
group_means = data.groupby("drawing_skills")[steps].mean()

# 有效性 ranking：按 beta_pos（或 beta）大小
eff_rank = beta_pos.sort_values(ascending=False).index.tolist()

print("\nEffectiveness rank (by beta_pos):", eff_rank)

print("\nGroup-level preference ranks vs effectiveness rank:")
for lvl, row in group_means.iterrows():
    pref_rank = row.sort_values(ascending=False).index.tolist()
    # 用 Spearman 相关（把 rank 转成序数）
    eff_order = {name: i for i, name in enumerate(eff_rank)}
    pref_order = {name: i for i, name in enumerate(pref_rank)}
    x = [eff_order[s] for s in steps]
    y = [pref_order[s] for s in steps]
    rho, p = spearmanr(x, y)
    print(f"ability_of_art={lvl}: preference rank={pref_rank} | Spearman rho={rho:.3f}, p={p:.3f}")

# =========================¥
# 6) 可视化（可选）：alignment vs positive_mood（按 ability 着色）
# =========================
plt.figure(figsize=(7,5))
for lvl, sub in data.groupby("drawing_skills"):
    plt.scatter(sub[ALIGN_COL], sub["positive_emotion"], s=12, alpha=0.55, label=f"ability={int(lvl)}")
plt.xlabel(ALIGN_COL)
plt.ylabel("positive_mood")
plt.title("Alignment vs Positive Mood (colored by art ability)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import KFold
from scipy.stats import zscore


df.columns = [c.strip() for c in df.columns]
df = df.replace({"na": np.nan, "NA": np.nan, "N/A": np.nan, "": np.nan, " ": np.nan})
df = df.dropna(how="all")

for c in df.columns:
    if c != "words":
        df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.rename(columns={"positive mood": "positive_mood"})

steps = ["writing_down_emotions", "imagination", "picture_reference", "color_expression"]
controls = ["ability_of_art", "difficulty", "engagement", "interest"]

need_cols = ["positive_mood"] + steps + controls
data = df[need_cols].dropna().reset_index(drop=True).copy()
print("N used:", len(data))

# ========= 1) 交叉拟合：每折训练估 E，测试算 alignment =========
kf = KFold(n_splits=5, shuffle=True, random_state=42)

alignment_oof = np.full(len(data), np.nan)  # out-of-fold alignment_index
E_oof = []  # 记录每折的E（可选，用于报告稳定性）

USE_CONTROLS_FOR_WEIGHTS = True  # True: 用 steps+controls 估 E；False: 只用 steps 估 E

for fold, (tr_idx, te_idx) in enumerate(kf.split(data), 1):
    train = data.loc[tr_idx].copy()
    test  = data.loc[te_idx].copy()

    # --- 1.1 在训练集估计 effectiveness betas（标准化回归系数） ---
    if USE_CONTROLS_FOR_WEIGHTS:
        cols_for_weights = ["positive_mood"] + steps + controls
        X_vars = steps + controls
    else:
        cols_for_weights = ["positive_mood"] + steps
        X_vars = steps

    ztr = train[cols_for_weights].apply(lambda s: (s - s.mean()) / s.std(ddof=0))
    formula = "positive_mood ~ " + " + ".join(X_vars)
    m = smf.ols(formula, data=ztr).fit()

    beta = m.params[steps]
    beta_pos = beta.clip(lower=0)
    if beta_pos.sum() == 0:
        # 极端情况下避免除0：退化成平均分配
        E = np.ones(len(steps)) / len(steps)
    else:
        E = (beta_pos / beta_pos.sum()).values

    E_oof.append(E)

    # --- 1.2 在测试集计算个人偏好权重 w_i，并算 alignment_index = w_i · E ---
    S = test[steps].copy()
    row_sum = S.sum(axis=1).replace(0, np.nan)
    W = S.div(row_sum, axis=0).fillna(0)

    alignment = W.values @ E
    alignment_oof[te_idx] = alignment

    print(f"Fold {fold}: E = {dict(zip(steps, np.round(E,4)))}")

data["alignment_cf"] = alignment_oof

# ========= 2) 用“跨折得到的 alignment_cf”做回归（更严谨） =========
# (R1) 仅 alignment
r1 = smf.ols("positive_mood ~ alignment_cf", data=data).fit()

# (R2) alignment + controls
r2 = smf.ols("positive_mood ~ alignment_cf + " + " + ".join(controls), data=data).fit()

print("\n[Cross-fitted R1] positive_mood ~ alignment_cf")
print(r1.summary().tables[1])

print("\n[Cross-fitted R2] positive_mood ~ alignment_cf + controls")
print(r2.summary().tables[1])

# ========= 3) 可选：看 E 在不同fold是否稳定 =========
E_df = pd.DataFrame(E_oof, columns=steps)
print("\nE stability across folds (mean ± std):")
print(pd.DataFrame({"mean": E_df.mean(), "std": E_df.std()}))

# ========= 4) 可选可视化：alignment_cf vs mood =========
import matplotlib.pyplot as plt
plt.figure(figsize=(7,5))
plt.scatter(data["alignment_cf"], data["positive_mood"], s=12, alpha=0.6)
plt.xlabel("alignment_cf (cross-fitted)")
plt.ylabel("positive_mood")
plt.title("Cross-fitted Alignment vs Positive Mood")
plt.tight_layout()
plt.show()
